# Can a seemingly random cortex self-organise?

Salt-and-pepper cortex looks a little like someone dropped an orientation map, swept up the pieces, and put them back at random 🧂. But what if the map is not missing? What if it is simply hiding?

**The short answer is: yes!** In this demo, self-organisation builds a fabric of tiny, interconnected cortical domains. The structure is obvious on a perfect lattice; move the neurons only a little, and the map appears to vanish—even though the learned representation is still there. The map has not disappeared. It has gone undercover.

We will follow the clues one at a time. If you just want the story, keep scrolling; if you want to look under the bonnet, open the **Technical details** blocks.

<details>
<summary><strong>Technical details</strong></summary>

This notebook is a complete training-and-presentation workflow for micro-GCAL. If the completed archive is absent, it trains a 100×100 V1 sheet for two epochs; otherwise it loads the archived run. Code cells contain configuration and public helper calls only. Collection, checkpointing, measurements, plotting, animation, synthetic probes, and displacement analyses live in `helpers/microdomain_demo.py`.

</details>


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'demo_microdomains' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from demo_microdomains.helpers.microdomain_demo import (
    animation_html,
    animate_dimensionality_learning,
    animate_map_learning,
    animate_rotating_umap_grid,
    animate_scattered_map_learning,
    animate_synthetic_learning,
    animate_weight_learning,
    animate_robustness_learning,
    ensure_github_assets,
    estimate_demo_archive_gib,
    github_demo_config,
    macaque_displacement_table,
    plot_lgn_inputs_and_statistics,
    plot_macaque_displacement_links,
    plot_macaque_displacement_summary,
    run_macaque_displacement_demo,
    train_or_load_demo,
)

## 0. First, meet our tiny cortex 🧠

We need a cortex small enough to fit inside a computer but complicated enough to surprise us. This notebook can grow one from scratch and then replay what happened. If a completed run is already waiting, we skip the several-hour origin story and jump straight to the interesting bits.

<details>
<summary><strong>Technical details</strong></summary>

The canonical configuration uses a 100×100 V1 microdomain sheet, 80×80 natural-image crops, two independently shuffled epochs, and 100 learning snapshots. `train_or_load_demo` treats a completed archive as immutable: it loads valid existing data and trains only when that archive is absent or incomplete.

</details>


In [ ]:
config = github_demo_config()
estimate_demo_archive_gib(config)

In [ ]:
archive = train_or_load_demo(config)
archive.manifest['status'], archive.n_frames

In [ ]:
assets = ensure_github_assets(archive)
assets

## 1. Meet micro-GCAL: local competition, distant cooperation

Before we feed the model any images, here is the cast of characters. Read the figure from the bottom up: a visual stimulus becomes a sparse, contrast-normalised pattern in the **LGN**, which sends feedforward drive to a recurrent sheet representing **V1 layer 4**. This is the only cortical layer in this demo, so from here on we simply call it **V1** or **cortex**.

Each V1 neuron receives four kinds of input:

- **Afferent input** from a small local patch of LGN. These plastic connections become the neuron's visual receptive field.
- **Short-range excitation (SRE)** from its nearest cortical neighbours. Nearby co-active neurons encourage one another, helping a small patch settle together.
- **Longer-range inhibition (LRI)** from a wider surrounding neighbourhood. This suppresses competing activity and stops one excited patch from swallowing the whole sheet.
- **Cross-domain excitation (CDE)** with the widest reach. Unlike a uniform excitatory halo, CDE is learned selectively: strong links grow mainly between neurons that are repeatedly active together, allowing separated but functionally related patches to cooperate.

The spatial ordering is therefore **SRE < LRI < CDE**: local cooperation, broader competition, and selective cooperation again at the longest scale. That last ingredient is the key twist. Local excitation and inhibition keep individual domains tiny; CDE couples those tiny domains into a coordinated fabric and helps their recurrent responses resist noise. Think **many small islands connected by learned bridges**, rather than one continent—or a random scattering of boats. 🏝️

<img src="demo_assets/microdomain/micro_gcal_architecture.png" alt="Micro-GCAL architecture with LGN input, a recurrent V1 sheet, short-range excitation, longer-range inhibition, and cross-domain excitation" width="760">

<details>
<summary><strong>Technical details</strong></summary>

Micro-GCAL belongs to the GCAL/LISSOM family of self-organising cortical-map models. For each stimulus, LGN activity provides a fixed afferent drive while V1 activity is updated recurrently for 30 settling steps. At each step, a neuron's net drive combines its afferent input, positive SRE, subtractive LRI, and positive CDE, followed by thresholding and a saturating nonlinearity.

After settling, non-negative Hebbian plasticity strengthens connections between co-active pre- and postsynaptic units. CDE uses a thresholded Hebbian rule, so appreciable long-range excitation develops only between strongly co-active neurons rather than everywhere inside its radius. Incoming weights are normalised, adaptive thresholds maintain sparse activity, and slow gain control balances afferent and recurrent contributions. Repeating this settle–learn cycle over natural-image patches jointly shapes receptive fields, recurrent connectivity, and the spatial map.

In the figure, red denotes excitation, blue denotes inhibition, and the patchiness of CDE illustrates its learned selectivity. White-to-black activity maps run from zero to maximum mean activity. The dashed line identifies one example V1 neuron and the interaction fields centred on it.

</details>


## 2. Give it something to look at 👀

Our model starts with little glimpses of the natural world. An LGN-like stage turns them into sparse patches of edges and texture, then hands them to V1. There are no orientation labels and no hidden answer sheet—the cortex has to work out the useful structure for itself.

<details>
<summary><strong>Technical details</strong></summary>

Natural-image crops are passed through an ON-centre Laplacian-of-Gaussian filter and divisive gain control. The fixed held-out examples below provide a common reference for intensity, exact-zero sparsity, spatial power, and the number of principal components needed to explain 95% of LGN variance.

</details>

<img src="demo_assets/microdomain/lgn_inputs.png" alt="LGN inputs and statistics" width="800">


In [ ]:
plot_lgn_inputs_and_statistics(archive)

## 3. Now let the neurons negotiate

Each new input starts a brief conversation among the neurons: excite nearby partners, inhibit competitors, settle, learn, repeat. Slowly, tiny orientation domains appear. They may look busy, but they are not random—the Fourier ring exposes their preferred spacing, while the retinotopic fishnet bends locally without losing the global plot.

<details>
<summary><strong>Technical details</strong></summary>

For each input, activity settles through short-range excitation, plastic longer-range inhibition, and selective cross-domain excitation before Hebbian and homeostatic updates. From left to right the animation shows orientation preference, its spatial Fourier power, horizontal receptive-field position, and the central quarter of the retinotopic fishnet. In the Fourier panel, black means power and white means none.

</details>

![Orientation map formation](demo_assets/microdomain/map_learning.gif)


In [ ]:
animation_html(animate_map_learning(archive))

## 4. Connections begin choosing their friends

No one paints the finished map onto the sheet. The incoming connections learn which visual patterns matter, while cross-domain excitation learns which distant patches tend to belong together. The result is not a collection of lonely dots, but a fabric of tiny domains that have started exchanging phone numbers.

<details>
<summary><strong>Technical details</strong></summary>

The left canvas follows sampled LGN→V1 afferent weights after independent zero-centred normalisation. The right canvas follows source-centred cross-domain excitatory fields. Only complete interior crops are used, with no border padding. Both use the same perceptually uniform scale with exact zero shown in white.

</details>

![Afferent and cross-domain plasticity](demo_assets/microdomain/weight_learning.gif)


In [ ]:
animation_html(animate_weight_learning(archive))

## 5. Can it remember a face? 🙂

A colourful map is not much use if it forgets what it saw. So we show the network the same synthetic face throughout learning and ask a decoder to rebuild it from cortical activity. The face makes progress easy to spot, while the final curve keeps us honest: it reports average fidelity across the full fixed evaluation set, not just our photogenic volunteer.

<details>
<summary><strong>Technical details</strong></summary>

At every snapshot a concurrently trained nonlinear decoder maps settled V1 activity back to LGN space. The first three panels show the fixed face, its cortical response, and its reconstruction. The last panel is the mean input–reconstruction cosine similarity across all held-out reconstruction examples—not the face score alone. No intermediate image is interpolated.

</details>

![Tracked reconstruction fidelity](demo_assets/microdomain/synthetic_learning.gif)


In [ ]:
animation_html(animate_synthetic_learning(archive))

## 6. The population finds a shortcut

Ten thousand neurons do not need ten thousand independent opinions. As learning proceeds, groups of neurons begin varying together, and PCA reveals that shared structure. V1 can then describe the input with fewer effective degrees of freedom—a compact code, rather than a lossy shrug.

<details>
<summary><strong>Technical details</strong></summary>

The first three panels show the spatial loadings of the leading principal components through learning. The final plot tracks the number of components required to explain 95% of V1 variance and compares it with the fixed LGN value. Their ratio is a unitless estimate of relative representational dimensionality—roughly, the fraction of input degrees of freedom retained by the cortical code.

</details>

![PCA geometry and dimensionality](demo_assets/microdomain/dimensionality.gif)


In [ ]:
animation_html(animate_dimensionality_learning(archive))

## 7. Add noise and see whether it panics ⚡

A compact code still needs to keep its story straight when conditions get messy. We inject noise during the recurrent conversation and compare the result with the clean response. Selective excitation helps the network return to nearly the same answer, so the code becomes not only efficient but dependable.

<details>
<summary><strong>Technical details</strong></summary>

The same fixed LGN input and the same noise realisation are used at every learning snapshot. The middle panels compare matched 20×20 windows of clean activity and activity with noise intensity 0.06. The final plot tracks the population cosine similarity between clean and noisy settled codes through training.

</details>

![Noise robustness](demo_assets/microdomain/robustness.gif)


In [ ]:
animation_html(animate_robustness_learning(archive))

## 8. Now shake the seating plan 🌀

Here comes the trick. Real neurons do not sit forever on a perfect square lattice; they disperse during development. Move each model neuron by only about two positions and local clustering survives, consistent with cellular-scale observations such as Ringach et al. (2016), but the global periodic signature melts away. We have not changed what the neurons learned—only where their chairs ended up.

<details>
<summary><strong>Technical details</strong></summary>

The seeded Gaussian local permutation used in `wiring_efficiency.ipynb` is calibrated to the requested mean displacement in lattice units and held fixed throughout the animation. It is applied consistently to orientation preference and receptive-field identity. The panels show the displaced orientation map, its Fourier power, horizontal retinotopy, and the zoomed fishnet. Set `scatter_displacement` below to change the mean displacement.

</details>

![Scatter permutation](demo_assets/microdomain/scattered_learning.gif)


In [ ]:
scatter_displacement = 2.0
animation_html(
    animate_scattered_map_learning(
        archive, mean_displacement=scatter_displacement
    )
)

## 9. Ask real cortex how far the chairs moved

Two model positions sounds modest, but is it biologically plausible? We cannot rewind cortical development, so we use dense macaque V1 measurements as a clue: compare each neuron's actual location with the nearest place where a smooth underlying map predicts its orientation. It is not a developmental GPS trace, but it gives us a useful scale for the missing journey.

<details>
<summary><strong>Technical details</strong></summary>

The analysis uses the three densest 850×850 µm fields (`MB_1`, `MB_2`, and `MA_2`) from the public Chen et al. macaque V1 cellular-orientation dataset. A leave-one-cell-out circular map is smoothed with a shared Gaussian σ of 100 µm. Every significantly tuned cell is shown, but contour matches above 350 µm are excluded from the reported mean and the linked subsample so rare extreme matches do not dominate the estimate.

| Dataset | Included cells | Mean displacement |
|---|---:|---:|
| `MB_1` | 792 | 91.4 µm |
| `MB_2` | 729 | 82.5 µm |
| `MA_2` | 395 | 92.3 µm |

</details>

<img src="demo_assets/microdomain/macaque_displacement_summary.png" alt="Macaque cellular scatter and smooth maps" width="1050">


In [ ]:
macaque_smoothing_sigma_um = 100.0
macaque_max_displacement_um = 350.0
macaque_results = run_macaque_displacement_demo(
    smoothing_sigma_um=macaque_smoothing_sigma_um,
    max_displacement_um=macaque_max_displacement_um,
)
plot_macaque_displacement_summary(macaque_results)
macaque_displacement_table(macaque_results)

### Draw a few of those journeys

A table of distances is useful; little journeys are easier to feel. Each black line asks: “if this neuron belonged to the smooth underlying map, where would its preferred orientation place it?”

<details>
<summary><strong>Technical details</strong></summary>

A reproducible sparse subset of included cells is overlaid on a faint smooth map. Each solid black segment starts at the measured soma and ends at the nearest exact same-orientation location predicted by the leave-one-cell-out map; a black × marks that endpoint. Only 20 examples per field are drawn for legibility, while the titles retain the included-cell means.

</details>

<img src="demo_assets/microdomain/macaque_displacement_links.png" alt="Sparse macaque displacement links" width="1050">


In [ ]:
macaque_link_examples = 20
plot_macaque_displacement_links(
    macaque_results, n_links=macaque_link_examples, map_alpha=0.28
)

## 10. Leave the sheet and look for the hidden shape ✨

The cortical map now looks scrambled, so let us stop judging the network by cortical position. In response space, the order reappears: gratings, the topological model, the salt-and-pepper model, and real mouse V1 all trace similarly smooth, folded geometries. The map can disappear from the sheet while its hidden shape survives in the code.

<details>
<summary><strong>Technical details</strong></summary>

The four panels show separately fitted 3D UMAPs of raw full-phase grating pixels, topographic-model responses, salt-and-pepper-model responses, and high-arousal responses from Stringer et al. (2021) random-phase recording 1. Colour denotes orientation modulo 180°. The stimulus and model embeddings are compatible with the twisted angle–phase identification of a Klein bottle; the mouse data lack matched phase labels, so their resemblance is qualitative rather than a formal topological test. Synchronized rotation exposes the full geometry without distorting the fitted axis scales.

</details>

<img src="demo_assets/microdomain/rotating_umap.gif" alt="Rotating four-panel 3D UMAP comparison" width="1050">


In [ ]:
umap_rotation_frames = 48
umap_animation_asset = assets['rotating_umap']

## Conclusion: not random—just undercover

We started with a cortex that looked as though its map had been lost. Instead, the model offers a different ending: self-organisation can build selective receptive fields, efficient codes, robust dynamics, and structured coupling at a very fine scale. Modest neuronal displacement then hides the tidy layout without undoing the learning.

So salt-and-pepper need not mean **structureless**. It may mean **beautifully organised, then very lightly shuffled**.

<details>
<summary><strong>Technical takeaway</strong></summary>

Micro-GCAL extends classical recurrent self-organisation with learned selective cross-domain excitation. On the ideal sheet it produces small coupled domains, smooth retinotopy, reduced effective dimensionality, improving reconstruction fidelity, and robust settled codes. Local positional disorder masks the high-frequency spatial signatures, while local tuning similarity and smooth population-response geometry remain measurable. This is a model-based hypothesis—not proof that every biological salt-and-pepper map develops this way—but it gives the hypothesis concrete, testable signatures.

</details>
